# OpenPack Temporal VLM Fine-Tuning
**Live Notebook Link:** [https://www.kaggle.com/code/aryanthakur23/notebook889757d9e8](https://www.kaggle.com/code/aryanthakur23/notebook889757d9e8)
**Environment:** Kaggle 2xT4 GPUs (32GB Total)


In [ ]:
# VRAM Budget Calculation (Required Format)
model_base_4bit    = 2.0   # GB — Your model at 4-bit
lora_adapters      = 0.3   # GB — LoRA rank + alpha
frames_per_clip    = 8     # Sampled frames per 5-sec clip
frame_tokens       = 256   # Visual tokens per frame (after vision encoder)
batch_size         = 2
token_hidden_dim   = 1536  # Your model's hidden size
activation_gb      = (frames_per_clip * frame_tokens * batch_size * token_hidden_dim * 2) / 1e9
# With gradient checkpointing: activation_gb * 0.4 (recomputed, not stored)
total_vram_gb      = model_base_4bit + lora_adapters + (activation_gb * 0.4)
print(f"Estimated VRAM: {total_vram_gb:.2f} GB")
# Must be < 16 GB for T4 or < 40 GB for A100


In [ ]:
!pip install -q -U "transformers==4.49.0" "accelerate==1.2.1" "trl==0.13.0" "peft==0.14.0" "bitsandbytes==0.45.2" datasets webdataset qwen-vl-utils


In [ ]:
import os
# Must be set before torch init in this session
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["ACCELERATE_MIXED_PRECISION"] = "fp16"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import inspect
import torch
from datasets import Dataset
from transformers import (
    AutoProcessor,
    Qwen2VLForConditionalGeneration,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"

assert torch.cuda.is_available(), "CUDA GPU is required"
print("Visible GPU count:", torch.cuda.device_count())
print("Using GPU:", torch.cuda.get_device_name(0))

# 1) 4-bit QLoRA config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=False,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

# 2) Load processor + model
processor = AutoProcessor.from_pretrained(MODEL_ID, use_fast=True)
processor.tokenizer.padding_side = "right"
model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map={"": 0},
    attn_implementation="eager",
)

# 3) K-bit prep + gradient checkpointing
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model.enable_input_require_grads()
model.gradient_checkpointing_enable()

# 4) LoRA adapters
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)

# Ensure no trainable BF16 tensors remain and keep trainables in FP32 for AMP scaler safety
for name, p in model.named_parameters():
    if p.requires_grad:
        if p.dtype == torch.bfloat16:
            p.data = p.data.to(torch.float32)
        elif p.dtype == torch.float16:
            p.data = p.data.to(torch.float32)

model.config.use_cache = False
trainable_dtypes = sorted({str(p.dtype) for p in model.parameters() if p.requires_grad})
print("Trainable dtypes:", trainable_dtypes)
assert "torch.bfloat16" not in trainable_dtypes

# 5) Dummy dataset for pipeline validation
dummy_dataset = Dataset.from_dict({
    "text": [f"OpenPack sample {i}: dominant_operation Tape, next Put Items." for i in range(100)]
})

# 6) Memory-constrained args
training_args = TrainingArguments(
    output_dir="./qwen-openpack-checkpoints",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="adamw_torch",
    learning_rate=2e-5,
    fp16=True,
    bf16=False,
    max_grad_norm=0.0,
    max_steps=200,
    logging_steps=10,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=3,
    dataloader_num_workers=0,
    remove_unused_columns=True,
    report_to="none",
)

# 7) TRL-version-safe trainer init
trainer_kwargs = {
    "model": model,
    "args": training_args,
    "train_dataset": dummy_dataset,
}
sig = inspect.signature(SFTTrainer.__init__).parameters
if "processing_class" in sig:
    trainer_kwargs["processing_class"] = processor.tokenizer
elif "tokenizer" in sig:
    trainer_kwargs["tokenizer"] = processor.tokenizer

if "formatting_func" in sig:
    trainer_kwargs["formatting_func"] = lambda ex: ex["text"]
elif "dataset_text_field" in sig:
    trainer_kwargs["dataset_text_field"] = "text"

trainer = SFTTrainer(**trainer_kwargs)

print("Starting strict memory-constrained fine-tuning...")
torch.cuda.empty_cache()
trainer.train(resume_from_checkpoint=False)
